# Load and Analyze NEPSE Data with NepSense

This notebook demonstrates how to:
1. Load NEPSE data from NepSense
2. Calculate basic statistics
3. Visualize price trends
4. Identify trading opportunities

## Step 1: Load Data

In [ ]:
import pandas as pd
import numpy as np
import duckdb
from pathlib import Path

# Option 1: Load from CSV
df = pd.read_csv("data/master/nepsense_prices.csv")
print(f"Loaded {len(df)} rows")
print(f"Symbols: {df['symbol'].unique()}")
print(df.head())

## Step 2: Filter and Analyze

In [ ]:
# Filter to one company
nabil = df[df['symbol'] == 'NABIL'].copy()
nabil['date'] = pd.to_datetime(nabil['date'])
nabil = nabil.sort_values('date')

# Calculate daily returns
nabil['returns'] = nabil['close'].pct_change() * 100

# Summary statistics
print(f"\n{len(nabil)} trading days for NABIL")
print(f"Price range: {nabil['close'].min():.0f} - {nabil['close'].max():.0f}")
print(f"Average daily return: {nabil['returns'].mean():.3f}%")
print(f"Volatility (std dev): {nabil['returns'].std():.3f}%")
print(f"Max gain: {nabil['returns'].max():.2f}%")
print(f"Max loss: {nabil['returns'].min():.2f}%")

## Step 3: Simple Moving Average Strategy

In [ ]:
# Calculate moving averages
nabil['ma10'] = nabil['close'].rolling(10).mean()
nabil['ma50'] = nabil['close'].rolling(50).mean()

# Generate signals (MA10 > MA50 = Buy)
nabil['signal'] = np.where(nabil['ma10'] > nabil['ma50'], 1, 0)
nabil['position'] = nabil['signal'].diff()

# Buy signals (1) and Sell signals (-1)
buys = nabil[nabil['position'] == 1]
sells = nabil[nabil['position'] == -1]

print(f"Buy signals: {len(buys)}")
print(f"Sell signals: {len(sells)}")
print(f"\nRecent signals:")
print(nabil[['date', 'close', 'ma10', 'ma50', 'signal']].tail(10))

## Step 4: Calculate Strategy Returns

In [ ]:
# Simple backtest: buy on signal, hold until next sell
nabil['strategy_returns'] = nabil['returns'] * nabil['signal'].shift(1).fillna(0)

# Cumulative returns
nabil['cum_market_returns'] = (1 + nabil['returns'] / 100).cumprod() * 100
nabil['cum_strategy_returns'] = (1 + nabil['strategy_returns'] / 100).cumprod() * 100

# Final performance
market_return = nabil['cum_market_returns'].iloc[-1] - 100
strategy_return = nabil['cum_strategy_returns'].iloc[-1] - 100

print(f"\n=== Performance Comparison ===")
print(f"Buy & Hold return: {market_return:.2f}%")
print(f"MA Strategy return: {strategy_return:.2f}%")
print(f"Outperformance: {strategy_return - market_return:.2f}%")

## Step 5: Data Quality Check

In [ ]:
# Check for data quality issues
print("Data Quality Summary:")
print(f"Missing values:\n{df.isnull().sum()}")
print(f"\nSource confidence:\n{df['source_confidence'].value_counts()}")
print(f"\nDate range: {df['date'].min()} to {df['date'].max()}")
print(f"Number of symbols: {df['symbol'].nunique()}")